In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Permulaan Qubit

<Accordion>
<AccordionItem title="Versi pakej">

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Apabila Circuit dilaksanakan pada unit pemprosesan kuantum (QPU) IBM&reg;, set semula tersirat biasanya dimasukkan pada permulaan Circuit untuk memastikan qubit dimulakan kepada sifar. Ini dikawal oleh bendera `init_qubits`, ditetapkan sebagai [pilihan pelaksanaan primitif](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

Walau bagaimanapun, ketidaksempurnaan dalam proses set semula boleh memperkenalkan ralat penyediaan keadaan. Untuk mengurangkan ralat, QPU juga memasukkan masa lengah pengulangan (atau `rep_delay`) antara Circuit. Setiap backend mempunyai `rep_delay` lalai yang berbeza, tetapi biasanya ditetapkan untuk mengimbangi kesetiaan set semula dengan jumlah masa pelaksanaan. Jalankan `backend.default_rep_delay` untuk mencari `rep_delay` lalai bagi QPU tertentu.

Kerana semua QPU IBM menggunakan pelaksanaan kadar pengulangan dinamik, anda boleh menukar `rep_delay` untuk setiap kerja. Circuit yang anda hantar dalam kerja primitif dibungkus bersama untuk dilaksanakan pada QPU. Circuit ini dilaksanakan dengan berulang ke atas Circuit untuk setiap shot yang diminta; pelaksanaan adalah mengikut lajur atas matriks Circuit dan shot, seperti yang ditunjukkan dalam rajah berikut.

![The first column represents shot 0.  The circuits are run in order from 0 through 3.  The second column represents shot 1.  The circuits are run in order from 0 through 3.  The remaining columns follow the same pattern. ](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Matriks pelaksanaan mengikut lajur")

Kerana `rep_delay` dimasukkan antara Circuit, setiap shot pelaksanaan menghadapi lengahan ini. Oleh itu, apabila anda menurunkan `rep_delay`, jumlah masa pelaksanaan QPU berkurang, dengan mengorbankan peningkatan kadar ralat penyediaan keadaan, seperti yang ditunjukkan oleh imej berikut:

![This image shows that as the `rep_delay` value is lowered, the state preparation error rate increases.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Lengahan pengulangan berbanding kadar ralat")

Jika anda menetapkan kedua-dua `rep_delay=0` dan `init_qubits=False`, Circuit "bergabung" bersama, kerana qubit akan bermula dalam keadaan akhir dari shot sebelumnya.

Perhatikan bahawa walaupun Circuit dalam kerja primitif dibungkus bersama untuk pelaksanaan QPU, tiada jaminan tentang urutan pelaksanaan Circuit dari PUB. Sebagai contoh, jika anda menghantar `pubs=[pub1, pub2]`, Circuit dari `pub1` mungkin tidak berjalan sebelum Circuit dari `pub2`. Juga tiada jaminan bahawa Circuit dari kerja yang sama akan berjalan sebagai satu kelompok pada QPU.

## Tentukan `rep_delay` untuk kerja primitif
> **Note:** Sentiasa sahkan julat `rep_delay` yang disokong untuk QPU tertentu yang anda gunakan. Nilai-nilai ini tidak sama untuk setiap QPU dan juga boleh berubah dari masa ke masa.
> 
> Sila ambil perhatian bahawa peningkatan dalam `rep_delay` akan memberi kesan langsung kepada masa pelaksanaan dan penggunaan kapasiti anda.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Langkah seterusnya
> **Tip:** - Cuba contoh dalam tutorial [Algoritma pengoptimuman anggaran kuantum (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - Semak cara [memulakan dengan Estimator](/guides/get-started-with-estimator).
>   - Semak cara [memulakan dengan Sampler](/guides/get-started-with-sampler).